# Notebook 29 - Competent-teacher robustness arm (G7)

**Threat addressed.** Every AWBIR number on the shallow arm is referenced to a dense anchor that escalates 59.8% of benign validation traffic to alerts, so a reviewer can reasonably ask whether the headline findings survive against a competent teacher. This arm rebuilds the shallow anchor to meet pre-registered competence criteria, re-derives and recalibrates the five 40% structures for that teacher, and repeats the recovery factorial and the normalisation-collapse audit against it.

**Pre-registered robustness claims** (stage 2, fixed before any result):

- C1 equalisation: recovery equalisation index for AWBIR at least 0.70 in both regimes
- C2 variance dominance: recovery-seed share of AWBIR variance exceeds the selection-method share
- C3 rank instability: mean pairwise rank correlation across seeds below 0.5
- C4 normalisation collapse recurs: at least one epoch with benign escalation above 10% that is healthy under batch statistics

A claim that does not hold is reported as a failure of robustness for that finding, not removed from the paper. If no teacher candidate meets the competence bar, the arm still runs and records `teacher_meets_prereg_criteria: false` rather than stopping.

**Staged execution.** `RUN_REGIMES` in stage 2 controls which recovery regimes run. `full_ce` is six passes over the full training set for each of 25 cells and is much the heavier half, so run `["minimal_ce"]` first, commit, then `["full_ce"]`. Stage 8 analyses only regimes with a complete five-by-five grid and reports `verdict_complete` accordingly.

**Stages.** 1 bootstrap, 2 pre-registration and run configuration, 3 data and model definition, 4 dual-mode evaluation helpers with runtime proofs that the batch-statistic probe is both non-destructive and discriminative, 5 train and accept the competent teacher (per-candidate caching, no work lost on interruption), 6 re-derive and calibrate the five 40% structures, 7 the factorial with a dual-mode audit at every epoch (resumable per cell), 8 verdict, 9 figures.

No test access at any point. GPU required.

In [ ]:
# Stage 1 - bootstrap
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, sys, json, itertools
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, yaml
import matplotlib.pyplot as plt

REPO = Path("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
assert REPO.exists(), f"repo not found: {REPO}"
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.saber.bridge_ciciot import load_bridge
from src.saber.taxonomy import ciciot2023_taxonomy, DEFAULT_COST_PROFILES
from src.saber.surgery import (enumerate_cnn1d_channel_groups, prune_cnn1d_channels,
                               profile_forward_flops, count_parameters)
from src.saber.metrics import full_model_audit, action_weighted_boundary_inversion_rate

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
R = REPO / "results/saber"
OUT = R / "29_competent_teacher_arm"
OUT.mkdir(parents=True, exist_ok=True)
print("repo:", REPO, "| device:", DEVICE)


In [ ]:
# Stage 2 - pre-registration
#
# Reviewer threat addressed: every AWBIR number on the shallow arm is referenced to a dense
# anchor that escalates 59.8% of benign validation traffic to alerts. This arm rebuilds the
# shallow anchor so that it is operationally competent on benign traffic, then repeats the
# recovery factorial and the normalisation-collapse audit against it.
#
# Nothing here re-opens a closed gate. The question is robustness of three already-reported
# findings, with criteria fixed before any result exists.

PREREG = {
    "arm": "G7_competent_teacher_robustness",
    "threat": ("shallow dense anchor escalates 59.8% of benign validation traffic; AWBIR is "
               "teacher-referenced, so shallow-arm conclusions may reflect a flawed reference"),
    "teacher_acceptance": {
        "benign_to_attack_rate": "<= 0.10 on validation",
        "binary_attack_recall": ">= 0.90 on validation",
        "fine_macro_f1": ">= 0.50 on validation (not materially worse than the 0.56 anchor)"},
    "design": ("5 selection methods x 5 recovery seeds x 2 regimes (minimal_ce, full_ce) at "
               "realised 40% on structures re-derived for THIS teacher; dual-mode BatchNorm "
               "audit at every epoch; no test access"),
    "robustness_claims_under_test": [
        "C1 equalisation: recovery equalisation index for AWBIR >= 0.70 in both regimes",
        "C2 variance dominance: recovery-seed share of AWBIR variance exceeds selection-method share",
        "C3 rank instability: mean pairwise rank correlation across seeds < 0.5 on AWBIR",
        "C4 normalisation collapse recurs: at least one epoch with benign_to_attack > 0.10 that "
        "is healthy under batch statistics"],
    "reporting_rule": ("each claim is reported as held or not held. A claim that does not hold "
                       "is reported as a failure of robustness for that finding, not removed."),
    "seeds": [101, 211, 307, 401, 503], "no_test_access": True,
}
(OUT / "G7_PREREGISTRATION.json").write_text(json.dumps(PREREG, indent=2))
print(json.dumps(PREREG, indent=2))

METHODS = ["random", "magnitude", "taylor", "fisher", "saber_v2"]
SEEDS = PREREG["seeds"]
REGIMES = {"minimal_ce": 1, "full_ce": 6}

# Staged execution. "full_ce" is 6 passes over the full training set for each of 25 cells,
# so it is by far the heavier half. Run ["minimal_ce"] first, commit, then ["full_ce"].
RUN_REGIMES = ["minimal_ce", "full_ce"]
assert set(RUN_REGIMES) <= set(REGIMES)

TARGET_FLOPS = 0.40
COLLAPSE_B2A = 0.10
SUBSET_FRACTION = 0.10


In [ ]:
# Stage 3 - data and a competent shallow anchor
TRAIN_LOADER, VAL_LOADER, _TEST_UNUSED, OLD_TEACHER, CLASS_NAMES = load_bridge()
taxonomy = ciciot2023_taxonomy(CLASS_NAMES)
robust_graph = pd.read_csv(R / "14_risk_graph/asvg_edges_robust.csv")
N_CLASSES = len(CLASS_NAMES)
MIN_W = int(yaml.safe_load(open(REPO / "config/saber.yaml"))["groups"]["minimum_remaining_per_layer"])

_benign = [i for i, c in enumerate(CLASS_NAMES) if "benign" in str(c).lower()]
assert len(_benign) == 1
BENIGN_IDX = int(_benign[0])
print("benign class index:", BENIGN_IDX, f"({CLASS_NAMES[BENIGN_IDX]})")


class CNN1D(nn.Module):
    """Same topology as the original shallow anchor: two conv blocks, 64 then 128 channels."""
    def __init__(self, n_classes=34):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(64),
            nn.Conv1d(64, 128, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(128))
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(128, n_classes)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        return self.head(self.pool(self.conv(x.float())).squeeze(-1))


# Evaluation subsample, shuffled so batch-statistic probing is meaningful.
Xv, Yv = VAL_LOADER.dataset.tensors
VAL_Y_ALL = Yv.numpy()
_rng = np.random.default_rng(0)
_idx = np.concatenate([_rng.permutation(np.where(VAL_Y_ALL == c)[0])[:4000]
                       for c in range(N_CLASSES) if (VAL_Y_ALL == c).sum() > 0])
_idx = np.random.default_rng(12345).permutation(_idx)
EX_X = Xv[_idx].to(DEVICE); EX_Y = VAL_Y_ALL[_idx]
EXAMPLE_INPUT = Xv[:8].float().to(DEVICE)

_train_y = TRAIN_LOADER.dataset.tensors[1].numpy()
COUNTS = np.bincount(_train_y, minlength=N_CLASSES)
N_TRAIN = len(TRAIN_LOADER.dataset)
print("evaluation rows:", len(_idx), "| benign share of train:",
      round(float(COUNTS[BENIGN_IDX] / COUNTS.sum()), 4))


def class_weights(alpha):
    """alpha=0.5 reproduces the original 1/sqrt(n) weighting; smaller alpha is milder."""
    w = np.zeros(N_CLASSES, dtype=np.float64)
    nz = COUNTS > 0
    w[nz] = 1.0 / np.power(COUNTS[nz], alpha)
    w[nz] /= w[nz].mean()
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)


In [ ]:
# Stage 4 - evaluation helpers (eval mode and batch-statistic probe)
def forward_logits(model):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(EX_X[i:i + 8192]).cpu()
                          for i in range(0, len(EX_X), 8192)]).numpy()


def forward_logits_batch_stats(model):
    saved = {n: (m.running_mean.clone(), m.running_var.clone(), m.momentum,
                 m.num_batches_tracked.clone())
             for n, m in model.named_modules()
             if isinstance(m, nn.BatchNorm1d) and m.running_mean is not None}
    model.train()
    for n, m in model.named_modules():
        if n in saved:
            m.momentum = 0.0
    with torch.no_grad():
        out = torch.cat([model(EX_X[i:i + 8192]).cpu()
                         for i in range(0, len(EX_X), 8192)]).numpy()
    for n, m in model.named_modules():
        if n in saved:
            rm, rv, mom, nbt = saved[n]
            m.running_mean.copy_(rm); m.running_var.copy_(rv)
            m.momentum = mom; m.num_batches_tracked.copy_(nbt)
    model.eval()
    return out


def audit_logits(logits, reference=None):
    a = full_model_audit(logits, EX_Y, taxonomy, DEFAULT_COST_PROFILES)
    if reference is not None:
        aw, _ = action_weighted_boundary_inversion_rate(reference, logits, EX_Y, robust_graph)
        a["awbir"] = float(aw)
    else:
        a["awbir"] = 0.0
    return a


def train_epochs(model, loader, epochs, weights, lr=1e-3, log=None, reference=None):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.CrossEntropyLoss(weight=weights)
    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            lossf(model(xb.float().to(DEVICE)), yb.to(DEVICE)).backward()
            opt.step()
        if log is not None:
            a_eval = audit_logits(forward_logits(model), reference)
            a_batch = audit_logits(forward_logits_batch_stats(model), reference)
            log.append({"epoch": ep,
                        "eval_b2a": float(a_eval["benign_to_attack_rate"]),
                        "eval_family_f1": float(a_eval["family_macro_f1"]),
                        "eval_fine_f1": float(a_eval["fine_macro_f1"]),
                        "eval_a2b": float(a_eval["attack_to_benign_rate"]),
                        "eval_awbir": float(a_eval["awbir"]),
                        "eval_hsr": float(a_eval["hsr_balanced_soc"]),
                        "eval_ece": float(a_eval["ece15"]),
                        "batch_b2a": float(a_batch["benign_to_attack_rate"]),
                        "batch_family_f1": float(a_batch["family_macro_f1"])})
    return model.eval()


# runtime proof that the probe cannot alter a model
_m = CNN1D(N_CLASSES).to(DEVICE).eval()
_b = {n: (x.running_mean.clone(), x.running_var.clone())
      for n, x in _m.named_modules() if isinstance(x, nn.BatchNorm1d)}
_e1 = forward_logits(_m); _p = forward_logits_batch_stats(_m); _e2 = forward_logits(_m)
for n, x in _m.named_modules():
    if isinstance(x, nn.BatchNorm1d):
        assert torch.equal(x.running_mean, _b[n][0]) and torch.equal(x.running_var, _b[n][1])
assert np.allclose(_e1, _e2, atol=1e-5), "probe altered the model output"
assert not _m.training, "model left in train mode"
# A probe that silently did nothing would report every collapse as healthy under batch
# statistics and so manufacture claim C4. Prove it actually differs from eval mode.
assert not np.allclose(_e1, _p, atol=1e-6), \
    "batch-statistic probe is indistinguishable from eval mode; C4 would be unfalsifiable"
print("batch-statistic probe verified non-destructive and discriminative")


In [ ]:
# Stage 5 - train the competent anchor and check it against the pre-registered criteria
CKPT = REPO / "models/ciciot2023/cnn1d_competent_seed0.pt"
CAND_DIR = REPO / "models/ciciot2023/competent_candidates"
CAND_DIR.mkdir(parents=True, exist_ok=True)
ALPHA_GRID = [0.10, 0.20, 0.30, 0.40, 0.50]   # 0.50 reproduces the original weighting
EPOCHS_TEACHER = 4
SEL_CSV = OUT / "teacher_selection.csv"


def meets(row):
    return (row["benign_to_attack_rate"] <= 0.10 and row["binary_attack_recall"] >= 0.90
            and row["fine_macro_f1"] >= 0.50)


if CKPT.exists():
    payload = torch.load(CKPT, map_location="cpu", weights_only=False)
    TEACHER = CNN1D(N_CLASSES)
    TEACHER.load_state_dict(payload["state_dict"])
    TEACHER = TEACHER.to(DEVICE).eval()
    CHOSEN_ALPHA = float(payload["alpha"])
    TEACHER_MEETS_PREREG = bool(payload.get("meets_criteria", True))
    print("competent teacher loaded from cache (alpha =", CHOSEN_ALPHA,
          "| meets criteria:", TEACHER_MEETS_PREREG, ")")
else:
    # Each candidate is cached, so an interruption never costs a retrain.
    rows = pd.read_csv(SEL_CSV).to_dict("records") if SEL_CSV.exists() else []
    seen = {float(r["alpha"]) for r in rows}
    for alpha in ALPHA_GRID:
        cand = CAND_DIR / f"alpha_{alpha:.2f}.pt"
        if alpha in seen and cand.exists():
            print(f"alpha={alpha}: cached, skipping"); continue
        torch.manual_seed(0); np.random.seed(0)
        m = train_epochs(CNN1D(N_CLASSES), TRAIN_LOADER, EPOCHS_TEACHER, class_weights(alpha))
        a = audit_logits(forward_logits(m))
        row = {"alpha": float(alpha),
               "benign_to_attack_rate": float(a["benign_to_attack_rate"]),
               "binary_attack_recall": float(a["binary_attack_recall"]),
               "fine_macro_f1": float(a["fine_macro_f1"]),
               "family_macro_f1": float(a["family_macro_f1"])}
        row["meets_criteria"] = bool(meets(row))
        rows = [r for r in rows if float(r["alpha"]) != alpha] + [row]
        torch.save({"state_dict": m.cpu().state_dict(), "alpha": float(alpha)}, cand)
        m = m.to(DEVICE)
        pd.DataFrame(rows).sort_values("alpha").to_csv(SEL_CSV, index=False)
        print(f"alpha={alpha}: b2a={row['benign_to_attack_rate']:.4f} "
              f"attack_recall={row['binary_attack_recall']:.4f} "
              f"fine_f1={row['fine_macro_f1']:.4f} -> "
              f"{'meets criteria' if row['meets_criteria'] else 'does not meet criteria'}")

    table = pd.DataFrame(rows).sort_values("alpha")
    passing = table[table["meets_criteria"]]
    if len(passing):
        chosen = passing.sort_values("fine_macro_f1", ascending=False).iloc[0]
        TEACHER_MEETS_PREREG = True
    else:
        # Report rather than crash: the arm still runs, and the verdict records that no
        # candidate met the pre-registered competence bar.
        viable = table[table["binary_attack_recall"] >= 0.90]
        pool = viable if len(viable) else table
        chosen = pool.sort_values("benign_to_attack_rate").iloc[0]
        TEACHER_MEETS_PREREG = False
        print("\nWARNING: no candidate met the pre-registered teacher criteria. "
              "Proceeding with the lowest benign-escalation candidate and recording "
              "teacher_meets_prereg = False in the verdict.")
    CHOSEN_ALPHA = float(chosen["alpha"])
    TEACHER = CNN1D(N_CLASSES)
    TEACHER.load_state_dict(torch.load(CAND_DIR / f"alpha_{CHOSEN_ALPHA:.2f}.pt",
                                       map_location="cpu", weights_only=False)["state_dict"])
    TEACHER = TEACHER.to(DEVICE).eval()
    torch.save({"state_dict": TEACHER.cpu().state_dict(), "alpha": CHOSEN_ALPHA,
                "meets_criteria": TEACHER_MEETS_PREREG}, CKPT)
    TEACHER = TEACHER.to(DEVICE).eval()

T_LOGITS = forward_logits(TEACHER)
T_AUDIT = audit_logits(T_LOGITS)
benign_recall = 1.0 - float(T_AUDIT["benign_to_attack_rate"])
print("\ncompetent teacher (alpha =", CHOSEN_ALPHA, "):",
      {k: round(float(T_AUDIT[k]), 4) for k in
       ["fine_macro_f1", "family_macro_f1", "benign_to_attack_rate",
        "attack_to_benign_rate", "binary_attack_recall"]})
print("benign recall:", round(benign_recall, 4),
      "| original anchor: fine_macro_f1 0.5596, benign_to_attack_rate 0.5977 "
      "(benign recall 0.4023)")
(OUT / "teacher_audit.json").write_text(json.dumps(
    {"alpha": CHOSEN_ALPHA, "meets_prereg_criteria": TEACHER_MEETS_PREREG,
     "benign_recall": benign_recall,
     "original_anchor_benign_to_attack_rate": 0.5977,
     "original_anchor_fine_macro_f1": 0.5596,
     **{k: float(v) for k, v in T_AUDIT.items()}}, indent=2))


In [ ]:
# Stage 6 - re-derive the five 40% structures for THIS teacher
groups = enumerate_cnn1d_channel_groups(TEACHER, EXAMPLE_INPUT)
conv_paths = sorted(groups["module_path"].unique())
mods = dict(TEACHER.named_modules())
print("prunable groups:", len(groups), "| layers:",
      groups.groupby("module_path")["group_id"].count().to_dict())

mag = {p: mods[p].weight.detach().pow(2).sum(dim=(1, 2)).sqrt().cpu().numpy() for p in conv_paths}
tay = {p: np.zeros(mods[p].weight.shape[0]) for p in conv_paths}
fis = {p: np.zeros(mods[p].weight.shape[0]) for p in conv_paths}
lossf = nn.CrossEntropyLoss(weight=class_weights(CHOSEN_ALPHA))
for nb, (xb, yb) in enumerate(VAL_LOADER):
    TEACHER.zero_grad()
    lossf(TEACHER(xb.float().to(DEVICE)), yb.to(DEVICE)).backward()
    for p in conv_paths:
        w, g = mods[p].weight, mods[p].weight.grad
        tay[p] += (w * g).sum(dim=(1, 2)).abs().detach().cpu().numpy()
        fis[p] += (w * g).pow(2).sum(dim=(1, 2)).detach().cpu().numpy()
    if nb >= 39:
        break
TEACHER.zero_grad()

# semantic boundary leverage, frozen V-C definition
sbl = {p: np.zeros(mods[p].weight.shape[0]) for p in conv_paths}
for _, e in robust_graph.iterrows():
    src, dst, w_e = int(e["source_index"]), int(e["target_index"]), float(e["robust_weight"])
    rows = np.where(EX_Y == src)[0][:256]
    if len(rows) == 0:
        continue
    TEACHER.zero_grad()
    z = TEACHER(EX_X[rows])
    (z[:, src] - z[:, dst]).mean().backward()
    for p in conv_paths:
        w, g = mods[p].weight, mods[p].weight.grad
        sbl[p] += w_e * (w * g).sum(dim=(1, 2)).abs().detach().cpu().numpy()
TEACHER.zero_grad()

scores = groups[["group_id", "module_path", "channel_index"]].copy()
for name, d in [("magnitude", mag), ("taylor", tay), ("fisher", fis), ("sbl_raw", sbl)]:
    scores[name] = [d[r.module_path][int(r.channel_index)] for r in scores.itertuples()]
scores["sem_rank"] = scores.groupby("module_path")["sbl_raw"].rank(pct=True)
scores["saber_v2"] = scores["sem_rank"] * scores["module_path"].map(
    scores.groupby("module_path")["fisher"].mean())
scores["random"] = np.random.default_rng(1).random(len(scores))
scores.to_csv(OUT / "teacher_scores.csv", index=False)

M0_FLOPS = profile_forward_flops(TEACHER, EXAMPLE_INPUT)["flops_per_item"]


def removal_order(column):
    left = {p: int((scores["module_path"] == p).sum()) for p in conv_paths}
    seq = []
    for r in scores.sort_values(column, ascending=True).itertuples():
        if left[r.module_path] - 1 < MIN_W:
            continue
        left[r.module_path] -= 1
        seq.append((r.module_path, int(r.channel_index)))
    return seq


def prune_prefix(seq, k):
    pm = {}
    for p, c in seq[:k]:
        pm.setdefault(p, []).append(c)
    st, _ = prune_cnn1d_channels(TEACHER, {p: sorted(cs) for p, cs in pm.items()},
                                 EXAMPLE_INPUT, minimum_remaining_per_layer=MIN_W)
    return st.to(DEVICE)


def realised(st):
    return 1 - profile_forward_flops(st, EXAMPLE_INPUT)["flops_per_item"] / M0_FLOPS


STRUCTURES, struct_rows = {}, []
for method in METHODS:
    seq = removal_order(method)
    lo, hi = 1, len(seq)
    while lo < hi:                                   # binary search to realised 40%
        mid = (lo + hi) // 2
        if realised(prune_prefix(seq, mid)) >= TARGET_FLOPS:
            hi = mid
        else:
            lo = mid + 1
    k = lo
    st = prune_prefix(seq, k)
    STRUCTURES[method] = seq[:k]
    struct_rows.append({"method": method, "k": k, "realised_flops": float(realised(st)),
                        "parameters": int(count_parameters(st))})
    print(f"{method}: k={k} realised={realised(st):.4f} params={count_parameters(st)}")
pd.DataFrame(struct_rows).to_csv(OUT / "structures.csv", index=False)
off = [r for r in struct_rows if abs(r["realised_flops"] - TARGET_FLOPS) > 0.015]
CALIBRATION_OK = not off
if off:
    print("WARNING: outside the 1.5-point calibration tolerance:", off)
    print("recorded in the verdict as calibration_within_tolerance = False")


def raw_student(method):
    pm = {}
    for p, c in STRUCTURES[method]:
        pm.setdefault(p, []).append(c)
    st, _ = prune_cnn1d_channels(TEACHER, {p: sorted(cs) for p, cs in pm.items()},
                                 EXAMPLE_INPUT, minimum_remaining_per_layer=MIN_W)
    return st.to(DEVICE)


In [ ]:
# Stage 7 - the factorial: 5 methods x 5 seeds x 2 regimes, dual-mode audit each epoch
RUNS = OUT / "factorial_runs.csv"
EPOCHS = OUT / "factorial_epochs.csv"


def complete(frame, key):
    if frame.empty:
        return set()
    return set(map(tuple, frame[key].drop_duplicates().values))


run_rows = pd.read_csv(RUNS).to_dict("records") if RUNS.exists() else []
ep_rows = pd.read_csv(EPOCHS).to_dict("records") if EPOCHS.exists() else []
done = complete(pd.DataFrame(run_rows), ["method", "regime", "seed"]) if run_rows else set()
print("complete cells:", len(done))

W = class_weights(CHOSEN_ALPHA)
for method in METHODS:
    for regime in RUN_REGIMES:
        n_ep = REGIMES[regime]
        for seed in SEEDS:
            if (method, regime, seed) in done:
                continue
            torch.manual_seed(seed); np.random.seed(seed)
            student = raw_student(method)
            a0 = audit_logits(forward_logits(student), T_LOGITS)

            gen = torch.Generator().manual_seed(seed)
            if regime == "minimal_ce":
                sub = torch.randperm(N_TRAIN, generator=gen)[: int(N_TRAIN * SUBSET_FRACTION)]
                ds = torch.utils.data.Subset(TRAIN_LOADER.dataset, sub.tolist())
            else:
                ds = TRAIN_LOADER.dataset
            loader = torch.utils.data.DataLoader(
                ds, batch_size=1024, shuffle=True,
                generator=torch.Generator().manual_seed(seed))

            log = []
            student = train_epochs(student, loader, n_ep, W, log=log, reference=T_LOGITS)
            for e in log:
                ep_rows.append({"method": method, "regime": regime, "seed": seed, **e})
            a = audit_logits(forward_logits(student), T_LOGITS)
            run_rows.append({"method": method, "regime": regime, "seed": seed,
                             "raw_awbir": float(a0["awbir"]),
                             "raw_family_f1": float(a0["family_macro_f1"]),
                             "raw_b2a": float(a0["benign_to_attack_rate"]),
                             "awbir": float(a["awbir"]),
                             "fine_macro_f1": float(a["fine_macro_f1"]),
                             "family_macro_f1": float(a["family_macro_f1"]),
                             "benign_to_attack_rate": float(a["benign_to_attack_rate"]),
                             "attack_to_benign_rate": float(a["attack_to_benign_rate"]),
                             "hsr_balanced_soc": float(a["hsr_balanced_soc"]),
                             "ece15": float(a["ece15"])})
            pd.DataFrame(run_rows).to_csv(RUNS, index=False)
            pd.DataFrame(ep_rows).to_csv(EPOCHS, index=False)
            print(f"{method} {regime} s{seed}: awbir={a['awbir']:.4f} "
                  f"famF1={a['family_macro_f1']:.4f} b2a={a['benign_to_attack_rate']:.4f}")

runs = pd.DataFrame(run_rows); epochs_df = pd.DataFrame(ep_rows)
print("runs:", len(runs), "| epoch rows:", len(epochs_df))


In [ ]:
# Stage 8 - verdict on the four robustness claims
runs = pd.read_csv(OUT / "factorial_runs.csv")
epochs_df = pd.read_csv(OUT / "factorial_epochs.csv")

def variance_shares(frame, metric):
    piv = frame.pivot_table(index="seed", columns="method", values=metric)
    x = piv.values
    grand = x.mean()
    ss_tot = ((x - grand) ** 2).sum()
    ss_m = len(piv) * ((x.mean(axis=0) - grand) ** 2).sum()
    ss_s = x.shape[1] * ((x.mean(axis=1) - grand) ** 2).sum()
    return {"method": float(ss_m / ss_tot), "seed": float(ss_s / ss_tot),
            "interaction_residual": float((ss_tot - ss_m - ss_s) / ss_tot)}

def rank_stability(frame, metric, lower_better=True):
    piv = frame.pivot_table(index="seed", columns="method", values=metric)
    ranks = piv.rank(axis=1, ascending=lower_better)
    cors = [ranks.loc[a].corr(ranks.loc[b], method="spearman")
            for a, b in itertools.combinations(ranks.index, 2)]
    return float(np.nanmean(cors)), [float(c) for c in cors]

counts = runs.groupby("regime").size()
ANALYSED = [r for r in REGIMES if counts.get(r, 0) == len(METHODS) * len(SEEDS)]
SKIPPED = [r for r in REGIMES if r not in ANALYSED]
if SKIPPED:
    print("SKIPPED (incomplete, rerun stage 7 with RUN_REGIMES set to these):", SKIPPED)
assert ANALYSED, "no regime has a complete method x seed grid yet"

per_regime = {}
for regime in ANALYSED:
    sub = runs[runs["regime"] == regime]
    raw_spread = sub.groupby("seed")["raw_awbir"].agg(lambda v: v.max() - v.min()).mean()
    rec_spread = sub.groupby("seed")["awbir"].agg(lambda v: v.max() - v.min()).mean()
    rei = float(1 - rec_spread / raw_spread) if raw_spread > 0 else float("nan")
    shares = variance_shares(sub, "awbir")
    mean_rho, cors = rank_stability(sub, "awbir")
    per_regime[regime] = {
        "recovery_equalisation_index_awbir": rei,
        "mean_raw_spread": float(raw_spread), "mean_recovered_spread": float(rec_spread),
        "variance_shares_awbir": shares,
        "mean_pairwise_rank_correlation_awbir": mean_rho,
        "pairwise_rank_correlations": cors,
        "mean_awbir_by_method": {m: float(v) for m, v in
                                 sub.groupby("method")["awbir"].mean().items()},
        "best_method_count": {m: int((sub.loc[sub.groupby("seed")["awbir"].idxmin(),
                                              "method"] == m).sum()) for m in METHODS}}

post = epochs_df.copy()
collapses = post[post["eval_b2a"] > COLLAPSE_B2A]
healthy_under_batch = collapses[collapses["batch_b2a"] <= COLLAPSE_B2A]
collapse_summary = {
    "epoch_evaluations": int(len(post)),
    "eval_mode_collapses": int(len(collapses)),
    "of_which_healthy_under_batch_statistics": int(len(healthy_under_batch)),
    "collapse_cells": [{"method": r.method, "regime": r.regime, "seed": int(r.seed),
                        "epoch": int(r.epoch), "eval_b2a": round(float(r.eval_b2a), 4),
                        "batch_b2a": round(float(r.batch_b2a), 4)}
                       for r in collapses.itertuples()]}

# A claim is only decided on complete regimes; partial runs cannot pass or fail it.
BOTH_REGIMES = set(ANALYSED) == set(REGIMES)
C1 = all(v["recovery_equalisation_index_awbir"] >= 0.70 for v in per_regime.values())
C2 = all(v["variance_shares_awbir"]["seed"] > v["variance_shares_awbir"]["method"]
         for v in per_regime.values())
C3 = all(v["mean_pairwise_rank_correlation_awbir"] < 0.5 for v in per_regime.values())
C4 = bool(len(healthy_under_batch) > 0)

verdict = {
    "arm": "G7_competent_teacher_robustness",
    "teacher": json.load(open(OUT / "teacher_audit.json")),
    "teacher_meets_prereg_criteria": bool(TEACHER_MEETS_PREREG),
    "calibration_within_tolerance": bool(CALIBRATION_OK),
    "regimes_analysed": ANALYSED,
    "verdict_complete": bool(BOTH_REGIMES),
    "C1_equalisation_holds": bool(C1),
    "C2_seed_variance_exceeds_method_variance": bool(C2),
    "C3_rank_instability_holds": bool(C3),
    "C4_normalisation_collapse_recurs": C4,
    "per_regime": per_regime,
    "collapse_audit": collapse_summary,
    "prereg": json.load(open(OUT / "G7_PREREGISTRATION.json")),
}
(OUT / "G7_verdict.json").write_text(json.dumps(verdict, indent=2))
print(json.dumps({k: v for k, v in verdict.items() if k not in ("prereg", "per_regime")}, indent=2))
for regime, v in per_regime.items():
    print(f"\n{regime}: REI={v['recovery_equalisation_index_awbir']:.3f} "
          f"shares(method/seed/inter)="
          f"{v['variance_shares_awbir']['method']:.3f}/"
          f"{v['variance_shares_awbir']['seed']:.3f}/"
          f"{v['variance_shares_awbir']['interaction_residual']:.3f} "
          f"rank_rho={v['mean_pairwise_rank_correlation_awbir']:.3f}")


In [ ]:
# Stage 9 - figures
runs = pd.read_csv(OUT / "factorial_runs.csv")
epochs_df = pd.read_csv(OUT / "factorial_epochs.csv")

fig, axes = plt.subplots(1, len(ANALYSED), figsize=(5.2 * len(ANALYSED), 3.4), squeeze=False)
for ax, regime in zip(axes[0], ANALYSED):
    sub = runs[runs["regime"] == regime]
    for method in METHODS:
        d = sub[sub["method"] == method].sort_values("seed")
        ax.plot(d["seed"].astype(str), d["awbir"], marker="o", lw=1.3, label=method)
    ax.set_title(f"competent teacher, {regime}")
    ax.set_xlabel("recovery seed"); ax.set_ylabel("AWBIR (lower is better)")
    ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "G7_awbir_by_seed.png", dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(6.4, 3.4))
v = json.load(open(OUT / "G7_verdict.json"))["per_regime"]
labels, bottoms = list(v), np.zeros(len(v))
for key, colour in [("method", "#4c72b0"), ("seed", "#dd8452"),
                    ("interaction_residual", "#55a868")]:
    vals = [v[r]["variance_shares_awbir"][key] for r in labels]
    ax.bar(labels, vals, bottom=bottoms, label=key, color=colour)
    bottoms += np.array(vals)
ax.set_ylabel("share of AWBIR variance"); ax.legend(fontsize=8)
ax.set_title("Competent teacher: variance decomposition")
fig.tight_layout(); fig.savefig(OUT / "G7_variance_decomposition.png", dpi=200); plt.show()

fig, axes = plt.subplots(1, len(ANALYSED), figsize=(5.2 * len(ANALYSED), 3.2), squeeze=False)
for ax, regime in zip(axes[0], ANALYSED):
    sub = epochs_df[epochs_df["regime"] == regime]
    for seed in SEEDS:
        d = sub[sub["seed"] == seed].groupby("epoch")["eval_b2a"].max()
        ax.plot(d.index, d.values, marker="o", lw=1.1, label=f"seed {seed}")
    ax.axhline(COLLAPSE_B2A, color="0.4", ls=":", lw=0.9)
    ax.set_yscale("symlog", linthresh=1e-3)
    ax.set_title(f"{regime}: worst benign escalation per epoch")
    ax.set_xlabel("epoch"); ax.set_ylabel("benign to attack rate"); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "G7_collapse_audit.png", dpi=200); plt.show()
print("figures written ->", OUT)
